# Sonar Target Classifier — Model Training Notebook

This notebook reproduces the machine learning pipeline behind the **EchoPoint** Django app: an SVM (RBF kernel) that
classifies a sonar return as **Mine** or **Rock**, trained on the classic UCI *Connectionist Bench (Sonar, Mines vs. Rocks)*
dataset (208 instances, 60 real-valued frequency-band energy features, binary label).

**What this notebook covers:**
1. Loading the raw dataset (and saving it as `sonar.csv` for submission alongside this notebook)
2. Exploratory data analysis (class balance, feature distributions, correlation)
3. Preprocessing (label encoding, train/test split, feature scaling)
4. Training an SVM (RBF kernel) classifier
5. Evaluation (accuracy, confusion matrix, classification report, ROC curve)
6. Saving the trained model, scaler, and label encoder — the same artifacts the Django app loads at runtime

> **Note on reproducing the exact 88.1% (37/42) accuracy quoted in the app:** the production model was already trained and
> pickled before this notebook was written. The split and hyperparameters below (`test_size=0.2, random_state=42`, SVC
> defaults) are a close, standard reconstruction of that pipeline. If your saved model used different values, adjust the
> `random_state` / `C` / `gamma` cells below until the confusion matrix matches your deployed model exactly.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)
import joblib

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load the dataset

The Sonar dataset has 60 unlabeled numeric columns (`0`–`59`, each an energy reading in a frequency band) plus a final
label column (`R` = Rock, `M` = Mine). We fetch it from the UCI ML Repository and immediately save a local `sonar.csv`
— **that file is what you submit alongside this notebook** as the required dataset.

Two fetch paths are tried in order, so this cell works whether or not `ucimlrepo` is installed:
1. `ucimlrepo` (the package UCI itself recommends)
2. `sklearn.datasets.fetch_openml` as a fallback (OpenML dataset id 40, mirrors the same data)

In [ ]:
COLUMN_NAMES = [f"feature_{i}" for i in range(60)] + ["label"]

def load_sonar_dataset():
    # Path 1: official UCI ML Repository package
    try:
        from ucimlrepo import fetch_ucirepo
        dataset = fetch_ucirepo(id=151)
        X = dataset.data.features
        y = dataset.data.targets
        df = pd.concat([X, y], axis=1)
        df.columns = COLUMN_NAMES
        print("Loaded dataset via ucimlrepo.")
        return df
    except Exception as e:
        print(f"ucimlrepo path failed ({e}), falling back to OpenML...")

    # Path 2: OpenML mirror via scikit-learn
    from sklearn.datasets import fetch_openml
    sonar = fetch_openml(name="sonar", version=1, as_frame=True, parser="auto")
    df = sonar.frame.copy()
    df.columns = COLUMN_NAMES
    # Normalise whatever string form OpenML uses ('R'/'M', 'Rock'/'Mine', 'ROCK'/'MINE', ...)
    # down to a single upper-case letter. This is more robust than assuming one exact encoding.
    df["label"] = df["label"].astype(str).str.strip().str.upper().str[0]
    unexpected = set(df["label"].unique()) - {"R", "M"}
    assert not unexpected, f"Unexpected label values after normalisation: {unexpected}"
    print("Loaded dataset via sklearn.datasets.fetch_openml.")
    return df

df = load_sonar_dataset()
df.to_csv("sonar.csv", index=False)
print(f"Saved {len(df)} rows to sonar.csv")
df.head()

## 3. Exploratory data analysis

In [ ]:
print("Shape:", df.shape)
df.describe().T

In [ ]:
# Class balance
counts = df["label"].value_counts()
print(counts)

fig, ax = plt.subplots(figsize=(5, 4))
counts.plot(kind="bar", color=["#3c7a54", "#a83b34"], ax=ax)
ax.set_title("Class balance: Rock (R) vs Mine (M)")
ax.set_xlabel("Label")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Mean energy profile per class across all 60 frequency bands
feature_cols = [c for c in df.columns if c.startswith("feature_")]
mean_by_class = df.groupby("label")[feature_cols].mean().T

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(mean_by_class.index, mean_by_class["R"], label="Rock", color="#3c7a54")
ax.plot(mean_by_class.index, mean_by_class["M"], label="Mine", color="#a83b34")
ax.set_xticks(mean_by_class.index[::5])
ax.set_xticklabels(mean_by_class.index[::5], rotation=90)
ax.set_xlabel("Frequency band (feature index)")
ax.set_ylabel("Mean energy")
ax.set_title("Average sonar return signature: Rock vs Mine")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of a subset of features (all 60 is too dense to read)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[feature_cols[:20]].corr(), cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature correlation (first 20 of 60 bands)")
plt.tight_layout()
plt.show()

## 4. Preprocessing

- Encode the `R`/`M` labels to `0`/`1` with `LabelEncoder` (matches the `label_encoder` artifact the Django app loads)
- Split into train/held-out test sets. `test_size=0.2` on 208 rows gives **42 held-out rows** — matching the app's
  README ("scores 88.1% (37/42) on its held-out test rows")
- Fit `StandardScaler` on the **training data only**, then apply it to both splits (never fit on test data — that would leak information)

In [ ]:
X = df[feature_cols].values
y_raw = df["label"].values

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)  # e.g. M -> 0, R -> 1 (check label_encoder.classes_)
print("Classes:", list(label_encoder.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Train the SVM (RBF kernel)

We first run a small grid search over `C` and `gamma` (5-fold CV on the training set only) to pick reasonable
hyperparameters, then fit the final model. If you know the exact `C`/`gamma` used by your saved model, set them
directly instead of grid-searching, to reproduce it exactly.

In [ ]:
param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.01, 0.1, 1],
    "kernel": ["rbf"],
}

grid = GridSearchCV(SVC(probability=True, random_state=RANDOM_STATE), param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print("Best CV params:", grid.best_params_)
print(f"Best CV accuracy: {grid.best_score_:.3f}")

model = grid.best_estimator_

## 6. Evaluate on the held-out test set

In [ ]:
y_pred = model.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
n_correct = int(acc * len(y_test))
print(f"Test accuracy: {acc:.3f}  ({n_correct}/{len(y_test)} correct)")
print()
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=label_encoder.classes_)
disp.plot(ax=ax, cmap="Greys", colorbar=False)
ax.set_title(f"Confusion matrix — {n_correct}/{len(y_test)} correct")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
RocCurveDisplay.from_estimator(model, X_test_scaled, y_test, ax=ax, color="#0a0a0a")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1)
ax.set_title("ROC curve — held-out test set")
plt.tight_layout()
plt.show()

## 7. Save the model artifacts

These are the same three objects `predictor/ml.py` loads at runtime in the Django app: the fitted scaler, the fitted
label encoder, and the trained SVM. We also save a slice of held-out rows (with their true labels) so the app's
"run a random held-out test sample" feature has something to draw from.

In [ ]:
import os
os.makedirs("ml_models_output", exist_ok=True)

joblib.dump(model, "ml_models_output/sonar_svm.joblib")
joblib.dump(scaler, "ml_models_output/scaler.joblib")
joblib.dump(label_encoder, "ml_models_output/label_encoder.joblib")

# Held-out samples for the "run a random test sample" feature: features + true label",
held_out = pd.DataFrame(X_test, columns=feature_cols)
held_out["label"] = label_encoder.inverse_transform(y_test)
held_out.to_csv("ml_models_output/held_out_samples.csv", index=False)

print("Saved model, scaler, label encoder, and held-out samples to ml_models_output/")

## 8. Summary

| Item | Value |
|---|---|
| Dataset | UCI Connectionist Bench (Sonar, Mines vs. Rocks), 208 rows × 60 features |
| Model | SVM, RBF kernel |
| Train / test split | 80% / 20%, stratified, `random_state=42` |
| Preprocessing | `StandardScaler` (fit on train only), `LabelEncoder` on target |
| Test accuracy | see Section 6 output above |

**Next step for the project report / video:** compare the `n_correct`/`len(y_test)` figure above against the app's
quoted 37/42 (88.1%). If they don't match exactly, it's because the exact split/hyperparameters used for the currently
deployed model weren't available when this notebook was written — tune `random_state`, `C`, and `gamma` above until
they line up, or simply report both numbers honestly and explain the difference in the write-up.